# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [1]:
from dataclasses import dataclass
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

../../../../../../assets/03-wine-quality/inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [20]:
# TODO: implementa la lectura del CSV.
# Pista: usa DATA_PATH.open(newline="", encoding="utf-8")
rows = []
row = {}

with DATA_PATH.open(newline="", encoding="utf-8") as dataset:
    df = list(csv.DictReader(dataset))
    print(f"El número de filas del csv es {len(df)}")
    print(f"Las columnas del csv son {list(df[0].keys())}")
    row = df[0]
    print(f"En la variable row hemos guardado {row}")


El número de filas del csv es 5
Las columnas del csv son ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
En la variable row hemos guardado {'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}


`df` es una lista de diccionarios {columna: valor}

In [16]:
df[:2]

[{'sample_id': 'red-001',
  'fixed_acidity': '7.4',
  'volatile_acidity': '0.7',
  'citric_acid': '0.0',
  'residual_sugar': '1.9',
  'chlorides': '0.076',
  'free_sulfur_dioxide': '11.0',
  'total_sulfur_dioxide': '34.0',
  'density': '0.9978',
  'ph': '3.51',
  'sulphates': '0.56',
  'alcohol': '9.4'},
 {'sample_id': 'red-002',
  'fixed_acidity': '7.8',
  'volatile_acidity': '0.88',
  'citric_acid': '0.0',
  'residual_sugar': '2.6',
  'chlorides': '0.098',
  'free_sulfur_dioxide': '25.0',
  'total_sulfur_dioxide': '67.0',
  'density': '0.9968',
  'ph': '3.2',
  'sulphates': '0.68',
  'alcohol': '9.8'}]

### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo?
    - `sample_id`. Es un mero índice.
2. ¿Cuántas características numéricas hay?
    - 11. Todas menos `sample_id`.
3. ¿Qué columna extra añadirías para probar el contrato?
    - Región de la uva (categórica) o uvas por litro de vino (numérica)
4. ¿Qué valor de `ph` sería claramente inválido?
    - Menor de 0 y mayor de 14.

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [62]:
class WineQualityRequestPydantic(BaseModel):
    # TODO: prohíbe campos extra con model_config.
    model_config = ConfigDict(extra='forbid')

    # TODO: declara las 11 características con Field(ge=..., le=...).
    fixed_acidity       : float = Field(ge=0, le=20)
    volatile_acidity    : float = Field(ge=0, le=2)
    citric_acid         : float = Field(ge=0, le=2)
    residual_sugar      : float = Field(ge=0, le=20)
    chlorides           : float = Field(ge=0, le=1)
    free_sulfut_dioxide : float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density             : float = Field(ge=0.98, le=1.01)
    ph                  : float = Field(ge=2.5, le=4.5)
    sulphates           : float = Field(ge=0, le=3)
    alcohol             : float = Field(ge=5, le=20)

# TODO: separa sample_id de la fila y valida solo las características del modelo.
model_row = df[0].copy()
del model_row['sample_id']
a = model_row
a
try:
    request = WineQualityRequestPydantic.model_validate(a)
except:
    print(1)
    print(a)

1
{'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}


### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [ ]:
# TODO: completa los casos y captura ValidationError.
# try:
#     ...
# except ValidationError as error:
#     print(error)

## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [ ]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    # TODO: declara las mismas 11 características.
    pass

# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.

### Decisión de diseño

Completa la frase:

> Usaría Pydantic en __________ porque __________. Usaría `dataclass` en __________ porque __________.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [ ]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
    # "fixed_acidity",
    # ...
)

# TODO: construye vector usando getattr(request, name).
# vector = [...]

### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
- ¿El clasificador detectaría necesariamente el error?
- ¿Cómo evitarías que cada script definiera un orden distinto?

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [ ]:
class WineQualityPredictionPydantic(BaseModel):
    # TODO: declara los campos de salida y sus restricciones.
    pass

# TODO: escribe un ejemplo de salida para red-001.

## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.